# Convolutional Autoencoder

In [1]:
import torch
print('PyTorch version:', torch.__version__)

PyTorch version: 2.11.0+cu128


In [2]:
%reload_ext watermark
%watermark -a 'Yasir Awais Butt' -v -p torch,torchvision -gitchangelog

Author: Yasir Awais Butt

Time: 08:46:23

Python implementation: CPython
Python version       : 3.12.3
IPython version      : 9.13.0

torch      : 2.11.0+cu128
torchvision: 0.26.0+cu128

Git hash: 2202699c5fd38af398e2682f289a0868b1b91f0e



In [11]:
import torch, torch.nn as nn
import matplotlib.pyplot as plt
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [12]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {DEVICE} device')
RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed(RANDOM_SEED)
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
NUM_EPOCHS = 10
transform = transforms.Compose([
    transforms.ToTensor(),
])
train_dataset = datasets.MNIST(root='../../data', train=True, transform=transform, download=True)
test_dataset = datasets.MNIST(root='../../data', train=False, transform=transform, download=True)
train_loader = DataLoader(train_dataset, 
                          batch_size=BATCH_SIZE, 
                          shuffle=True, 
                          num_workers=2)
test_loader = DataLoader(test_dataset, 
                         batch_size=BATCH_SIZE, 
                         shuffle=False,
                         num_workers=2)

Using cuda device


In [13]:
set_deterministic = True
set_all_seeds = RANDOM_SEED
if set_deterministic:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
if set_all_seeds is not None:
    torch.manual_seed(set_all_seeds)
    torch.cuda.manual_seed_all(set_all_seeds)   

In [14]:
print('Training Set:\n')
for images, labels in train_loader:
    print('Image batch dimensions:', images.size())
    print('Image label dimensions:', labels.size())
    print(labels[:10])
    break
print('Testing Set:\n')
for images, labels in test_loader:
    print('Image batch dimensions:', images.size())
    print('Image label dimensions:', labels.size())
    print(labels[:10])
    break

Training Set:

Image batch dimensions: torch.Size([32, 1, 28, 28])
Image label dimensions: torch.Size([32])
tensor([1, 2, 8, 5, 2, 6, 9, 9, 9, 4])
Testing Set:

Image batch dimensions: torch.Size([32, 1, 28, 28])
Image label dimensions: torch.Size([32])
tensor([7, 2, 1, 0, 4, 1, 4, 9, 5, 9])


## Model

In [15]:
class Reshape(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.shape = args
    def forward(self, x):
        return x.view(self.shape)
    
class Trim(nn.Module):
    def __init__(self, *args):
        super().__init__()
    def forward(self, x):
        return x[:, :, :28, :28]
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),  # (B, 16, 28, 28)
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1), # (B, 32, 28, 28)
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1), # (B, 64, 14, 14)
            nn.LeakyReLU(0.01),
            nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1), # (B, 64, 7, 7)
            nn.LeakyReLU(0.01),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1), # (B, 64, 7, 7)
            nn.LeakyReLU(0.01),
            #Reshape(-1, 64*7*7), # (B, 3136)
            nn.Flatten(), # (B, 3136)
            nn.Linear(64*7*7, 2) # (B, 2)
        )
        self.decoder = nn.Sequential(
            nn.Linear(2, 64*7*7), # (B, 1568)
            Reshape(-1, 64, 7, 7), # (B, 64, 7, 7)
            nn.ConvTranspose2d(64, 64, kernel_size=3, stride=1, padding=1), # (B, 64, 7, 7)
            nn.LeakyReLU(0.01),
            nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1), # (B, 64, 7, 7)
            nn.LeakyReLU(0.01),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2), # (B, 32, 15, 15)
            Trim(), # (B, 32, 28, 28)
            nn.LeakyReLU(0.01),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2), # (B, 16, 29, 29)
            nn.LeakyReLU(0.01),
            nn.ConvTranspose2d(16, 1, kernel_size=3, stride=1), # (B, 1, 29, 29)
            Trim(), # (B, 1, 28, 28)
            nn.Sigmoid()
        )
    def forward(self,x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [16]:
model = ConvAutoencoder().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.MSELoss()

In [17]:
import time

In [18]:
start_time = time.time()
log_dict = {'train_loss_per_batch': [],
                'train_loss_per_epoch': []}
for epoch in range(NUM_EPOCHS):
    model.train()
    train_loss = 0
    batch_errors = []
    prev_batch_loss = None
    for batch_idx, (images, _) in enumerate(train_loader):
        images = images.to(DEVICE)
        x_hat = model(images)
        loss = criterion(x_hat, images)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        log_dict['train_loss_per_batch'].append(loss.item())
        print(f'Epoch [{epoch+1}/{NUM_EPOCHS}], '
              f'Batch [{batch_idx+1}/{len(train_loader)}], '
              f'Train Loss: {loss.item():.4f}', end='\r')
    train_loss /= len(train_loader.dataset)
    log_dict['train_loss_per_epoch'].append(train_loss)
    print(f'\nEpoch [{epoch+1}/{NUM_EPOCHS}], Train Loss: {train_loss:.4f}')

Epoch [1/10], Batch [1875/1875], Train Loss: 0.0570
Epoch [1/10], Train Loss: 0.0655
Epoch [2/10], Batch [1875/1875], Train Loss: 0.0587
Epoch [2/10], Train Loss: 0.0531
Epoch [3/10], Batch [1875/1875], Train Loss: 0.0512
Epoch [3/10], Train Loss: 0.0512
Epoch [4/10], Batch [1875/1875], Train Loss: 0.0466
Epoch [4/10], Train Loss: 0.0495
Epoch [5/10], Batch [1875/1875], Train Loss: 0.0496
Epoch [5/10], Train Loss: 0.0476
Epoch [6/10], Batch [1875/1875], Train Loss: 0.0447
Epoch [6/10], Train Loss: 0.0461
Epoch [7/10], Batch [1875/1875], Train Loss: 0.0360
Epoch [7/10], Train Loss: 0.0448
Epoch [8/10], Batch [1875/1875], Train Loss: 0.0408
Epoch [8/10], Train Loss: 0.0439
Epoch [9/10], Batch [1875/1875], Train Loss: 0.0435
Epoch [9/10], Train Loss: 0.0433
Epoch [10/10], Batch [1875/1875], Train Loss: 0.0405
Epoch [10/10], Train Loss: 0.0427
